# K-Sil adaptive $\tau$ vs K-Sil fixed $\tau=1$ vs K-Sil fixed $\tau=2$

### K-Sil temperature ablation setup

#### ksil

In [13]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances
from joblib import Parallel, delayed

class KSil:
    def __init__(self,
                 n_clusters=3,
                 init='random',
                 max_iter=100,
                 random_state=0,
                 init_temperature=1.0,
                 learning_rate=0.2,
                 tol=1e-4,
                 n_init=1,
                 n_jobs=1,
                 adaptive_temperature=True):

        # Parameters
        self.n_clusters = int(n_clusters)     # Number of clusters
        self.init = init                      # Initialization method
        self.max_iter = int(max_iter)         # Maximum number of iterations
        self.init_temperature = float(init_temperature) # Initial temperature
        self.random_state = int(random_state) # Random seed
        self.learning_rate = learning_rate    # Learning rate for temperature
        self.tol = float(tol)                 # Centroid convergence tolerance
        self.adaptive_temperature = adaptive_temperature # Use adaptive temperature

        # For multiple initializations:
        self.n_init = n_init                  # Number of initializations
        self.n_jobs = n_jobs                  # Number of cores

        # n_clusters check
        if self.n_clusters < 2:
           raise ValueError(f"n_clusters ({self.n_clusters}) must be > 1.")
        # init check
        if isinstance(self.init, str) and self.init not in ('random', 'k-means++'):
           raise ValueError(f"init ({self.init}) must be 'random' or 'k-means++'.")
        # n_init check
        if self.n_init < 1:
           raise ValueError(f"n_init ({self.n_init}) must be > 0.")

        # Attributes
        self.labels_ = None          # Cluster labels (converged partition)
        self.cluster_centers_ = None # Cluster centers (converged partition)
        self.n_iter_ = None          # Number of iterations (until convergence)
        self.sil_ = None             # silhouette (centroid-approximated) at converged partition

        # history attributes
        self.centers_history_ = None # Centroids across iterations
        self.labels_history_ = None  # Assignments across iterations
        self.sil_history_ = None     # Silhouette proxy across iterations
        self.weights_history_ = None # Weights across iterations

    def _initialization(self, X, n_clusters):
        X = np.asarray(X)
        if X.shape[0] < n_clusters:
           raise ValueError(f"n_clusters ({n_clusters}) can not exceed n_samples ({X.shape[0]}).")

        kmeans = KMeans(n_clusters=n_clusters,
                        init=self.init,
                        random_state=self.random_state,
                        n_init=1,
                        max_iter=1).fit(X)
        centers = kmeans.cluster_centers_
        labels = kmeans.labels_

        # Retry if some clusters are empty in the initial assignment
        if np.unique(labels).size < n_clusters:
           max_retries = 10 # 10 retries at max
           base_seed = self.random_state
           for attempt in range(1, max_retries + 1):
               seed = base_seed + attempt
               km = KMeans(n_clusters=n_clusters,
                           init=self.init,
                           random_state=seed,
                           n_init=1,
                           max_iter=1).fit(X)
               centers = km.cluster_centers_
               labels = km.labels_
               if np.unique(labels).size == n_clusters:
                  break
           else:
               raise ValueError(
                     f"KMeans (1-iter) initialization produced empty clusters after {max_retries} retries. "
                     f"Try a different random_state, or init='k-means++'.")
        return centers, labels

    def _fit_once(self, X, n_clusters, previous_centers, w):
        X = np.asarray(X)
        w = np.asarray(w)
        km = KMeans(
             n_clusters=n_clusters,
             init=previous_centers,
             n_init=1,
             max_iter=1,
             random_state=self.random_state)
        km.fit(X, sample_weight=w)

        centers = km.cluster_centers_
        labels = km.labels_
        return centers, labels

    @staticmethod
    def sil_scores(X, labels, centers):
        X = np.asarray(X, dtype=float)
        labels = np.asarray(labels, dtype=int)
        centers = np.asarray(centers, dtype=float)

        if len(X) == 0:
           raise ValueError("X is empty. Cannot compute silhouette scores.")

        n = X.shape[0]

        # Squared distances to all centroids
        D = euclidean_distances(X, centers, squared=True)

        # Distance to own centroid (a: intra cluster distance proxy)
        D_diag = D[np.arange(n), labels]  # a^2
        a_vals = np.sqrt(D_diag)  # a

        # Nearest other centroid distance (b: inter cluster distance)
        D_others = D.copy()
        D_others[np.arange(n), labels] = np.inf
        b_sq = D_others.min(axis=1)  # b^2
        b_vals = np.sqrt(b_sq)  # b

        # Silhouette surrogate (b-a)/[max{a,b}+epsilon]
        max_ab = np.maximum(np.maximum(a_vals, b_vals), 1e-12)
        s_vals = (b_vals - a_vals) / max_ab

        return s_vals, a_vals, b_vals

    @staticmethod
    def micro_sil(s_vals):
        s_vals = np.asarray(s_vals, dtype=float)
        if s_vals.size == 0:
           return 0.0
        return float(s_vals.mean())

    @staticmethod
    def macro_sil(s_vals, labels):
        s_vals = np.asarray(s_vals, dtype=float)
        labels = np.asarray(labels, dtype=int)

        if s_vals.size == 0 or labels.size == 0:
           return 0.0

        uniq, inv = np.unique(labels, return_inverse=True)
        if uniq.size == 0:
           return 0.0
        sum_per_cluster = np.bincount(inv, weights=s_vals)
        cnt_per_cluster = np.bincount(inv)

        valid = cnt_per_cluster > 0
        if not np.any(valid):
           return 0.0
        cluster_means = sum_per_cluster[valid] / cnt_per_cluster[valid]
        return float(cluster_means.mean())

    def _weights(self, s_vals, temperature):
        s_vals = np.asarray(s_vals, dtype=float)
        weights = np.exp(s_vals * temperature)
        return weights

    def _KSil(self, X, n_clusters, max_iter):

        # Initialize centroids
        centers, labels = self._initialization(X, n_clusters)
        tau = self.init_temperature # initial temperature
        prev_score =  None

        centers_history = [centers.copy()]
        labels_history = [labels.copy()]
        sil_history = []
        weights_history = []

        n_iter = 0

        while n_iter < max_iter:

            n_iter += 1

            # Compute point-silhouette scores
            sil_vals, a_vals, b_vals = KSil.sil_scores(X, labels, centers)
            score = KSil.macro_sil(sil_vals, labels)

            micro_sil = KSil.micro_sil(sil_vals)
            sil_history.append(micro_sil)

            # Update temperature
            if self.adaptive_temperature and prev_score is not None:
               r = (score - prev_score) / max(1 - prev_score, 1e-12)

               r = float(np.clip(r, -1.0, 1.0))

               # data-driven bounds to avoid softmax saturation
               counts = np.bincount(labels, minlength=n_clusters)
               m_max = int(max(counts.max(), 2))
               L     = float(np.sqrt(2.0 * np.log(m_max)))
               z_max = (m_max - 1) / m_max
               tau_min, tau_max = 1e-12, L / max(z_max, 1e-8)

               # multiplicative update (no logs needed)
               eta = self.learning_rate # default: 0.2 (small stable learning rate)

               # Update temperature based on the rate of change of the silhouette objective
               tau = float(np.clip(tau * np.exp(eta * r), tau_min, tau_max))

            # store for next iteration’s drift calc
            prev_score = score

            # Retain previous centroids for convergence checking
            previous_centers = centers.copy()

            # Compute weights based on silhouette scores
            weights = self._weights(sil_vals, tau)

            weights_history.append(weights.copy())

            # Update centroids and assignments
            centers, labels = self._fit_once(X, n_clusters, previous_centers, weights)

            centers_history.append(centers.copy())
            labels_history.append(labels.copy())

            avg_move = np.linalg.norm(centers - previous_centers, axis=1).mean()

            # Centroid stability
            if avg_move < self.tol:
               break

        return centers, labels, n_iter, sil_history, centers_history, labels_history, weights_history

    def fit(self, X):
        """
        Run KSil on X and store the converged partition.
        X: array-like, shape (n_samples, n_features)
        """
        X_arr = X.values if hasattr(X, "values") else np.asarray(X, dtype=float)

        if self.n_init == 1:
           (centers,
           labels,
           n_iter,
           sil_history,
           centers_history,
           labels_history,
           weights_history) = self._KSil(X_arr, self.n_clusters, self.max_iter)
        else:
           base_seed = self.random_state
           seeds = [int(base_seed) + i for i in range(self.n_init)]

           # Helper: directly call _KSil with a seed
           def _run_one(seed):
               self.random_state = seed
               return self._KSil(X_arr, self.n_clusters, self.max_iter)

           if self.n_jobs in (None, 1):
              results = [_run_one(seed) for seed in seeds]
           else:
              results = Parallel(n_jobs=self.n_jobs)(delayed(_run_one)(seed) for seed in seeds)

           # Pick best run by final silhouette
           best_idx = None
           best_sil = -np.inf

           for i, res in enumerate(results):
               (_, _, _, sil_history_i,
               _, _, _) = res
               if sil_history_i:
                  final_sil_i = sil_history_i[-1]
               else:
                  final_sil_i = -np.inf
               if final_sil_i > best_sil:
                   best_sil = final_sil_i
                   best_idx = i

           (centers,
           labels,
           n_iter,
           sil_history,
           centers_history,
           labels_history,
           weights_history) = results[best_idx]

           self.random_state = base_seed # restore base random_state

        self.cluster_centers_ = centers
        self.labels_ = labels
        self.n_iter_ = n_iter

        self.sil_history_ = sil_history
        self.sil_ = sil_history[-1] if sil_history else None
        self.centers_history_ = centers_history
        self.labels_history_ = labels_history
        self.weights_history_ = weights_history

        return self

    def predict(self, X):
        """
        Assign new points to the nearest learned centroid.
        """
        if self.cluster_centers_ is None:
           raise ValueError("KSil model is not fitted yet. Call '.fit(...)' first.")

        X_arr = X.values if hasattr(X, "values") else np.asarray(X, dtype=float)
        dist_matrix = euclidean_distances(X_arr, self.cluster_centers_, squared=True)
        labels = np.argmin(dist_matrix, axis=1)
        return labels.astype(int)

    def transform(self, X):
        """
        Return distances of each sample to each centroid.
        """
        if self.cluster_centers_ is None:
           raise ValueError("KSil model is not fitted yet. Call '.fit(...)' first.")

        X_arr = X.values if hasattr(X, "values") else np.asarray(X, dtype=float)
        distances = euclidean_distances(X_arr, self.cluster_centers_, squared=False)
        return distances

    def fit_predict(self, X):
        """
        Equivalent to: fit(X); return labels_.
        """
        self.fit(X)
        return self.labels_

    def fit_transform(self, X):
        """
        Equivalent to: fit(X); return transform(X).
        """
        self.fit(X)
        return self.transform(X)


# TEMPERATURE ABLATION — DIAGNOSTIC INSTRUMENTATION
#
# This subclass records the temperature used to compute
# instance weights at each iteration.
#
# This instrumentation is intended to diagnose whether
# adaptive temperature differs meaningfully from fixed
# temperature during optimization.

import numpy as np


class KSilTemperatureTracked(KSil):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.temperature_history_ = []
        self.temperature_final_ = None

    def _weights(self, s_vals, temperature):

        # Record the temperature actually used for weighting.
        self.temperature_history_.append(float(temperature))

        # Delegate the weight computation to the original KSil.
        return super()._weights(s_vals, temperature)

    def fit(self, X):

        # Reset the trace before each fit.
        self.temperature_history_ = []
        self.temperature_final_ = None

        super().fit(X)

        if self.temperature_history_:
            self.temperature_final_ = self.temperature_history_[-1]

        return self

#### Imports, helpers, datasets

##### Imports

In [14]:
!pip install ucimlrepo
import copy
import json
import math
import random
import re
import sys
import time
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import wilcoxon
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from sklearn.base import BaseEstimator, ClusterMixin
from sklearn.covariance import LedoitWolf
from sklearn.datasets import (
    make_blobs,
    load_iris,
    load_wine,
    load_digits,
    load_breast_cancer,
    fetch_openml,
    fetch_20newsgroups,
)
from sklearn.decomposition import PCA
import umap
from sklearn.impute import SimpleImputer
from sklearn.manifold import TSNE
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    silhouette_samples,
    silhouette_score,
    davies_bouldin_score,
    adjusted_rand_score,
    adjusted_mutual_info_score,
    pairwise_distances,
)
from sklearn.metrics import normalized_mutual_info_score as NMI
from sklearn.neighbors import NearestNeighbors, LocalOutlierFactor
from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    PowerTransformer,
    Normalizer,
    LabelEncoder,
    normalize,
)
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from joblib import Parallel, delayed
from ucimlrepo import fetch_ucirepo
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)
from huggingface_hub import login
from getpass import getpass

import warnings
warnings.filterwarnings(
    "ignore",
    message=".*Duplicate values are leading to incorrect results.*",
    category=UserWarning
)

from google.colab import userdata
# Get HF token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise ValueError("No secret named 'HF_TOKEN' found in Colab.")
login(token = hf_token.strip())

##### Silhouette Helper

In [15]:
#  Helper computation methods
class Silhouette:
    @staticmethod
    def Micro(X, labels):
        return silhouette_score(X, labels)

    @staticmethod
    def Macro(X, labels):
        silhouette_vals = silhouette_samples(X, labels)
        unique_labels = np.unique(labels)
        cluster_means = [
            np.mean(silhouette_vals[labels == lbl]) for lbl in unique_labels
        ]
        return np.mean(cluster_means) if cluster_means else 0

    @staticmethod
    def MacroApprox(X, labels, centers=None):
        X = np.asarray(X)
        y = np.asarray(labels).ravel()

        if X.ndim != 2:
            raise ValueError("X must be a 2D array of shape (n_samples, n_features).")
        n_samples, n_features = X.shape
        if n_samples == 0:
            return 0.0

        unique_labels, inv = np.unique(y, return_inverse=True)
        k = unique_labels.size
        if k < 2:
            return 0.0

        if centers is None:
            counts = np.bincount(inv, minlength=k).astype(float)
            sums = np.zeros((k, n_features), dtype=float)
            np.add.at(sums, inv, X)
            centers_mat = sums / counts[:, None]
        else:
            centers = np.asarray(centers, dtype=float)
            if centers.ndim != 2 or centers.shape[1] != n_features:
                raise ValueError(f"`centers` must have shape (k, d) with d={n_features}.")
            if centers.shape[0] != k:
                raise ValueError(f"`centers` first dimension must equal number of clusters k={k}.")
            if not np.array_equal(unique_labels, np.arange(k)):
                raise ValueError(
                    "When passing `centers` as ndarray, cluster labels must be 0..k-1 "
                    "so that centers[row] corresponds to that label."
                )
            centers_mat = centers

        D_sq = euclidean_distances(X, centers_mat, squared=True)

        a = np.sqrt(D_sq[np.arange(n_samples), inv])
        D_sq[np.arange(n_samples), inv] = np.inf
        b = np.sqrt(D_sq.min(axis=1))

        denom = np.maximum(np.maximum(a, b), 1e-12)
        s_point = (b - a) / denom

        cluster_sizes = np.bincount(inv, minlength=k).astype(float)
        s_point[cluster_sizes[inv] == 1] = 0.0

        sum_by_cluster = np.bincount(inv, weights=s_point, minlength=k)
        mean_by_cluster = sum_by_cluster / np.maximum(cluster_sizes, 1.0)
        return float(mean_by_cluster.mean())

    @staticmethod
    def MicroApprox(X, labels, centers=None):
        X = np.asarray(X)
        y = np.asarray(labels).ravel()

        if X.ndim != 2:
            raise ValueError("X must be a 2D array of shape (n_samples, n_features).")
        n_samples, n_features = X.shape
        if n_samples == 0:
            return 0.0

        unique_labels, inv = np.unique(y, return_inverse=True)
        k = unique_labels.size
        if k < 2:
            return 0.0

        if centers is None:
            counts = np.bincount(inv, minlength=k).astype(float)
            sums = np.zeros((k, n_features), dtype=float)
            np.add.at(sums, inv, X)
            centers_mat = sums / counts[:, None]
        else:
            centers = np.asarray(centers, dtype=float)
            if centers.ndim != 2 or centers.shape[1] != n_features:
                raise ValueError(f"`centers` must have shape (k, d) with d={n_features}.")
            if centers.shape[0] != k:
                raise ValueError(f"`centers` first dimension must equal number of clusters k={k}.")
            if not np.array_equal(unique_labels, np.arange(k)):
                raise ValueError(
                    "When passing `centers` as ndarray, cluster labels must be 0..k-1 "
                    "so that centers[row] corresponds to that label."
                )
            centers_mat = centers

        D_sq = euclidean_distances(X, centers_mat, squared=True)

        a = np.sqrt(D_sq[np.arange(n_samples), inv])
        D_sq[np.arange(n_samples), inv] = np.inf
        b = np.sqrt(D_sq.min(axis=1))

        denom = np.maximum(np.maximum(a, b), 1e-12)
        s_point = (b - a) / denom

        return float(s_point.mean())

##### Leukemia

In [16]:
data_leuk = fetch_openml(name="leukemia", version=1, as_frame=False)
X_leuk_raw = data_leuk.data
y_leuk_txt = data_leuk.target

le = LabelEncoder()
y_leuk = le.fit_transform(np.asarray(y_leuk_txt))

imputer = SimpleImputer(strategy='mean')
X_leuk_imputed = imputer.fit_transform(X_leuk_raw)

scaler = StandardScaler()
X_leuk_scaled = scaler.fit_transform(X_leuk_imputed)

X_leuk = X_leuk_scaled

k_leuk = len(np.unique(y_leuk))
name_leuk = "Leukemia"

print(f"Dataset: {name_leuk}")
print(f"Shape: {X_leuk.shape}")
print(f"K: {k_leuk}")

Dataset: Leukemia
Shape: (72, 7129)
K: 2


##### Wine

In [17]:
from sklearn.datasets import load_iris, load_wine
from sklearn.preprocessing import StandardScaler
import pandas as pd

data = load_wine()
X_wine = data.data
y_wine = data.target
X_wine = StandardScaler().fit_transform(X_wine)

##### Breast Cancer

In [18]:
from sklearn.datasets import load_iris, load_wine, load_breast_cancer
from sklearn.preprocessing import StandardScaler
import pandas as pd

data = load_breast_cancer()
X_bc = data.data
y_bc = data.target

X_bc = StandardScaler().fit_transform(X_bc)

##### SMS

In [19]:
encoder_minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def build_minilm_pca_features(
    texts,
    n_components=50,
    encoder=encoder_minilm,
    random_state=0,
):
    """
    texts -> MiniLM embeddings -> PCA(n_components) -> L2-normalized
    """
    X_emb = encoder.encode(
        list(map(str, texts)),
        convert_to_numpy=True,
        show_progress_bar=True
    ).astype(np.float64)

    pca = PCA(
        n_components=n_components,
        whiten=False,
        random_state=random_state
    )
    X_pca = pca.fit_transform(X_emb)

    normalizer = Normalizer(norm="l2", copy=False)
    X_final = normalizer.fit_transform(X_pca)

    return X_final, pca, normalizer

ds_sms = load_dataset("ucirvine/sms_spam", split="train")

texts_sms  = list(map(str, ds_sms["sms"]))
labels_sms_raw = list(ds_sms["label"])

le_sms = LabelEncoder()
y_sms = le_sms.fit_transform(labels_sms_raw)
k_sms = len(np.unique(y_sms))

print(f"SMS Spam: {len(texts_sms)} msgs, {k_sms} classes (ham/spam)")

X_sms_minilm, pca_sms, norm_sms_emb = build_minilm_pca_features(
    texts_sms,
    n_components=100,
    encoder=encoder_minilm,
    random_state=0,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SMS Spam: 5574 msgs, 2 classes (ham/spam)


Batches:   0%|          | 0/175 [00:00<?, ?it/s]

##### R8

In [20]:
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
import numpy as np

ds_r8 = load_dataset("dxgp/R8")
texts_r8 = ds_r8["train"]["text"]
labels_r8 = ds_r8["train"]["label"]

le = LabelEncoder()
y_r8 = le.fit_transform(labels_r8)
k_r8 = len(np.unique(y_r8))
name_r8 = "R8"

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

X_emb_r8 = model.encode(
    list(texts_r8),
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False,
)

print(f"Embeddings shape before PCA: {X_emb_r8.shape}")

pca = PCA(n_components=20, random_state=0)
X_r8_pca = pca.fit_transform(X_emb_r8)
y_r8 = y_r8

print(f"Dataset: {name_r8}, Shape after PCA: {X_r8_pca.shape}, k: {k_r8}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/155 [00:00<?, ?it/s]

Embeddings shape before PCA: (4936, 384)
Dataset: R8, Shape after PCA: (4936, 20), k: 8


##### Banking77

In [21]:
import numpy as np
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder, Normalizer
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

ds_train = load_dataset("PolyAI/banking77", split="train", revision="refs/convert/parquet")
ds_test  = load_dataset("PolyAI/banking77", split="test",  revision="refs/convert/parquet")

texts_b77 = list(map(str, ds_train["text"])) + list(map(str, ds_test["text"]))
labels_b77_raw = list(ds_train["label"]) + list(ds_test["label"])

le = LabelEncoder()
y_b77 = le.fit_transform(labels_b77_raw)

k_b77 = len(np.unique(y_b77))
name_b77 = "BANKING77"

encoder_b77 = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
X_b77_embed = encoder_b77.encode(
    texts_b77,
    convert_to_numpy=True,
    show_progress_bar=False
).astype(np.float64)

pca_b77 = PCA(
    n_components=50,
    whiten=False,
    random_state=0
)
X_b77_pca = pca_b77.fit_transform(X_b77_embed)

normalizer = Normalizer(norm="l2", copy=False)
X_b77 = normalizer.fit_transform(X_b77_pca)

print(f"Dataset: {name_b77}")
print(f"Shape: {X_b77.shape}")
print(f"K (true classes): {k_b77}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dataset: BANKING77
Shape: (13083, 50)
K (true classes): 77


In [22]:
pca_b77 = PCA(
    n_components=20,
    whiten=False,
    random_state=0
)

X_b77_pca = pca_b77.fit_transform(X_b77_embed)

normalizer = Normalizer(norm="l2", copy=False)
X_b77 = normalizer.fit_transform(X_b77_pca)

##### CLINC150

In [23]:
import numpy as np
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder, Normalizer
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

ds_train = load_dataset("DeepPavlov/clinc150", split="train")
ds_val   = load_dataset("DeepPavlov/clinc150", split="validation")
ds_test  = load_dataset("DeepPavlov/clinc150", split="test")

texts_clinc = (
    list(map(str, ds_train["utterance"])) +
    list(map(str, ds_val["utterance"])) +
    list(map(str, ds_test["utterance"]))
)

labels_clinc_raw = (
    list(ds_train["label"]) +
    list(ds_val["label"]) +
    list(ds_test["label"])
)

le_clinc = LabelEncoder()
y_clinc = le_clinc.fit_transform(labels_clinc_raw)

k_clinc = len(np.unique(y_clinc))
name_clinc = "CLINC150"

encoder_clinc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

X_clinc_embed = encoder_clinc.encode(
    texts_clinc,
    convert_to_numpy=True,
    show_progress_bar=True
).astype(np.float64)

pca_clinc = PCA(
    n_components=50,
    whiten=False,
    random_state=0
)
X_clinc_pca = pca_clinc.fit_transform(X_clinc_embed)

normalizer_clinc = Normalizer(norm="l2", copy=False)
X_clinc = normalizer_clinc.fit_transform(X_clinc_pca)

print(f"Dataset: {name_clinc}")
print(f"Shape: {X_clinc.shape}")
print(f"K (true classes): {k_clinc}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/744 [00:00<?, ?it/s]

Dataset: CLINC150
Shape: (23800, 50)
K (true classes): 151


##### MICE

In [24]:
from ucimlrepo import fetch_ucirepo

import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, Normalizer
from sklearn.impute import SimpleImputer
import umap

mice_protein_expression = fetch_ucirepo(id=342)

X_full = mice_protein_expression.data.features
y_full = mice_protein_expression.data.targets


X_mice_df = X_full.select_dtypes(include=[np.number])

if "class" in y_full.columns:
    y_mice_txt = y_full["class"].values
else:
    y_mice_txt = y_full.iloc[:, -1].values

le = LabelEncoder()
y_mice = le.fit_transform(y_mice_txt)

imputer = SimpleImputer(strategy='mean')
X_mice_imputed = imputer.fit_transform(X_mice_df)

scaler = StandardScaler()
X_mice = scaler.fit_transform(X_mice_imputed)

k_mice = len(np.unique(y_mice))
name_mice = "Mice Protein"

reducer_mice = umap.UMAP(
    n_components=10,
    n_neighbors=15,
    min_dist=0.0,
    metric="euclidean",
    random_state=0
)

X_mice_umap = reducer_mice.fit_transform(X_mice)

print(f"Dataset: {name_mice}")
print(f"Shape: {X_mice_umap.shape}")
print(f"K: {k_mice}")

/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Dataset: Mice Protein
Shape: (1080, 10)
K: 8


### Ablation

In [26]:
import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from scipy.sparse import issparse
from sklearn.metrics import adjusted_rand_score
from IPython.display import display

SEEDS = range(10, 20)
TEMPERATURES = [1.0, 2.0]

DATASET_VARIABLES = [
    ("Wine",          "X_wine",         "y_wine"),
    ("Breast Cancer", "X_bc",           "y_bc"),
    ("Leukemia",      "X_leuk",         "y_leuk"),
    ("BANKING77",     "X_b77",          "y_b77"),
    ("SMS",           "X_sms_minilm",   "y_sms"),
    ("CLINC150",      "X_clinc",        "y_clinc"),
    ("R8",            "X_r8_pca",       "y_r8"),
    ("Mice Protein",  "X_mice_umap",    "y_mice")
]


def to_numpy(X):
    if issparse(X):
        return X.toarray()
    if hasattr(X, "to_numpy"):
        return X.to_numpy()
    return np.asarray(X)


missing = [
    f"{name}: {variable}"
    for name, x_name, y_name in DATASET_VARIABLES
    for variable in (x_name, y_name)
    if variable not in globals()
]

if missing:
    raise NameError(
        "Missing dataset variables:\n" + "\n".join(missing)
    )


prepared = []

for name, x_name, y_name in DATASET_VARIABLES:
    X = np.asarray(to_numpy(globals()[x_name]), dtype=float)
    y = np.asarray(globals()[y_name]).ravel()

    if X.ndim != 2:
        raise ValueError(f"{name}: X must be 2D.")

    if len(X) != len(y):
        raise ValueError(f"{name}: X/y length mismatch.")

    if not np.isfinite(X).all():
        raise ValueError(f"{name}: X contains non-finite values.")

    k = len(np.unique(y))
    prepared.append((name, X, y, k))

assert len(prepared) == 8


def clustering_acc(y_true, y_pred):
    _, yt = np.unique(
        np.asarray(y_true).ravel(),
        return_inverse=True
    )

    _, yp = np.unique(
        np.asarray(y_pred).ravel(),
        return_inverse=True
    )

    contingency = np.zeros(
        (yt.max() + 1, yp.max() + 1),
        dtype=np.int64
    )

    np.add.at(contingency, (yt, yp), 1)

    rows, cols = linear_sum_assignment(
        contingency,
        maximize=True
    )

    return float(
        contingency[rows, cols].sum() / len(yt)
    )


def evaluate_model(model, X, y):
    labels = np.asarray(model.labels_)

    sil_values, _, _ = KSil.sil_scores(
        X,
        labels,
        model.cluster_centers_
    )

    sil_values = np.asarray(
        sil_values,
        dtype=float
    ).ravel()

    if len(sil_values) != len(labels):
        raise ValueError("Silhouette length mismatch.")

    if not np.isfinite(sil_values).all():
        raise ValueError("Non-finite silhouette scores.")

    micro = float(np.mean(sil_values))

    macro = float(np.mean([
        np.mean(sil_values[labels == c])
        for c in np.unique(labels)
    ]))

    return {
        "ARI": float(adjusted_rand_score(y, labels)),
        "ACC": clustering_acc(y, labels),
        "Micro_Proxy_SIL": micro,
        "Macro_Proxy_SIL": macro
    }


METRICS = [
    "ARI",
    "ACC",
    "Micro_Proxy_SIL",
    "Macro_Proxy_SIL"
]

rows = []

total_fits = (
    len(prepared)
    * len(TEMPERATURES)
    * len(SEEDS)
    * 2
)

completed = 0

for name, X, y, k in prepared:

    for tau0 in TEMPERATURES:

        for seed in SEEDS:

            results = {}

            for adaptive in (False, True):

                model = KSil(
                    n_clusters=k,
                    random_state=seed,
                    init_temperature=tau0,
                    adaptive_temperature=adaptive,
                    learning_rate=0.2,
                    max_iter=100,
                    tol=1e-4,
                    n_init=1
                )

                model.fit(X)

                key = (
                    "Adaptive" if adaptive else "Fixed"
                )

                results[key] = evaluate_model(
                    model,
                    X,
                    y
                )

                completed += 1

            row = {
                "Dataset": name,
                "τ": tau0,
                "Seed": seed
            }

            for metric in METRICS:
                row[f"Δ{metric}"] = (
                    results["Adaptive"][metric]
                    - results["Fixed"][metric]
                )

            rows.append(row)

        print(
            f"Completed {name:<16} | "
            f"τ={tau0:.0f} | "
            f"{completed}/{total_fits} fits"
        )


seed_deltas = pd.DataFrame(rows)

final_table = (
    seed_deltas
    .groupby(
        ["Dataset", "τ"],
        as_index=False,
        sort=False
    )
    [[f"Δ{metric}" for metric in METRICS]]
    .mean()
)

final_table = final_table.rename(
    columns={
        "ΔMicro_Proxy_SIL": "ΔMicro Proxy SIL",
        "ΔMacro_Proxy_SIL": "ΔMacro Proxy SIL"
    }
)

final_table = final_table[
    [
        "Dataset",
        "τ",
        "ΔARI",
        "ΔACC",
        "ΔMicro Proxy SIL",
        "ΔMacro Proxy SIL"
    ]
]


pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 250)

display(
    final_table.style.format({
        "τ": "{:.0f}",
        "ΔARI": "{:+.6e}",
        "ΔACC": "{:+.6e}",
        "ΔMicro Proxy SIL": "{:+.6e}",
        "ΔMacro Proxy SIL": "{:+.6e}"
    })
)

Completed Wine             | τ₀=1 | 20/320 fits
Completed Wine             | τ₀=2 | 40/320 fits
Completed Breast Cancer    | τ₀=1 | 60/320 fits
Completed Breast Cancer    | τ₀=2 | 80/320 fits
Completed Leukemia         | τ₀=1 | 100/320 fits
Completed Leukemia         | τ₀=2 | 120/320 fits
Completed BANKING77        | τ₀=1 | 140/320 fits
Completed BANKING77        | τ₀=2 | 160/320 fits
Completed SMS              | τ₀=1 | 180/320 fits
Completed SMS              | τ₀=2 | 200/320 fits
Completed CLINC150         | τ₀=1 | 220/320 fits
Completed CLINC150         | τ₀=2 | 240/320 fits
Completed R8               | τ₀=1 | 260/320 fits
Completed R8               | τ₀=2 | 280/320 fits
Completed Mice Protein     | τ₀=1 | 300/320 fits
Completed Mice Protein     | τ₀=2 | 320/320 fits


,Dataset,τ₀,ΔARI,ΔACC,ΔMicro Proxy SIL,ΔMacro Proxy SIL
0,Wine,1,+0.000000e+00,+0.000000e+00,+2.741913e-04,+2.598975e-04
1,Wine,2,+0.000000e+00,+0.000000e+00,+4.507498e-04,+4.245525e-04
2,Breast Cancer,1,+5.810435e-04,+1.757469e-04,+2.453655e-04,+2.267830e-04
3,Breast Cancer,2,+0.000000e+00,+0.000000e+00,+3.911615e-04,+3.299130e-04
4,Leukemia,1,+0.000000e+00,+0.000000e+00,+2.483342e-05,+2.703172e-05
5,Leukemia,2,+4.907712e-03,+2.777778e-03,+5.517067e-04,+3.382612e-04
6,BANKING77,1,+7.836086e-04,+2.094321e-03,+9.950147e-04,+1.332052e-03
7,BANKING77,2,+1.350032e-03,+2.415348e-03,+8.881051e-04,+1.124789e-03
8,SMS,1,+9.546675e-04,+3.767492e-04,+1.618650e-04,+1.495031e-04
9,SMS,2,+1.423915e-04,+5.382131e-05,+1.561361e-04,+1.284403e-04


In [29]:
temperature_rows = []

for name, X, y, k in prepared:
    for tau in TEMPERATURES:
        for seed in SEEDS:

            model = KSilTemperatureTracked(
                n_clusters=k,
                random_state=seed,
                init_temperature=tau,
                adaptive_temperature=True,
                learning_rate=0.2,
                max_iter=100,
                tol=1e-4,
                n_init=1
            )

            model.fit(X)

            final_tau = model.temperature_final_

            if final_tau is None:
                raise ValueError(
                    f"No temperature recorded for {name}, τ={tau}, seed={seed}."
                )

            temperature_rows.append({
                "Dataset": name,
                "τ": tau,
                "Seed": seed,
                "Final τ": final_tau,
                "Δτ": final_tau - tau,
                "Δτ (%)": 100 * (final_tau - tau) / tau
            })

        print(f"Completed {name:<16} | τ={tau:.0f}")

temperature_seed_results = pd.DataFrame(temperature_rows)

temperature_table = (
    temperature_seed_results
    .groupby(["Dataset", "τ"], as_index=False, sort=False)
    .agg({
        "Final τ": "mean",
        "Δτ": "mean",
        "Δτ (%)": "mean"
    })
)

display(
    temperature_table.style.format({
        "τ": "{:.0f}",
        "Final τ": "{:.6f}",
        "Δτ": "{:+.6f}",
        "Δτ (%)": "{:+.2f}%"
    })
)

Completed Wine             | τ=1
Completed Wine             | τ=2
Completed Breast Cancer    | τ=1
Completed Breast Cancer    | τ=2
Completed Leukemia         | τ=1
Completed Leukemia         | τ=2
Completed BANKING77        | τ=1
Completed BANKING77        | τ=2
Completed SMS              | τ=1
Completed SMS              | τ=2
Completed CLINC150         | τ=1
Completed CLINC150         | τ=2
Completed R8               | τ=1
Completed R8               | τ=2
Completed Mice Protein     | τ=1
Completed Mice Protein     | τ=2


,Dataset,τ,Final τ,Δτ,Δτ (%)
0,Wine,1,1.040179,+0.040179,+4.02%
1,Wine,2,2.084612,+0.084612,+4.23%
2,Breast Cancer,1,1.024120,+0.024120,+2.41%
3,Breast Cancer,2,2.054683,+0.054683,+2.73%
4,Leukemia,1,1.005836,+0.005836,+0.58%
5,Leukemia,2,2.009937,+0.009937,+0.50%
6,BANKING77,1,1.028488,+0.028488,+2.85%
7,BANKING77,2,2.063848,+0.063848,+3.19%
8,SMS,1,1.014250,+0.014250,+1.43%
9,SMS,2,2.031140,+0.031140,+1.56%
